In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#add checkpoints  Google Drive

Mounted at /content/drive


In [ ]:
import zipfile, os
# Update this path
zip_path = "/content/drive/MyDrive/results.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/")
    print("Extracted:", z.namelist())

Extracted: ['.config/.last_opt_in_prompt.yaml', '.config/.last_update_check.json', '.config/.last_survey_prompt.yaml', '.config/config_sentinel', '.config/gce', '.config/default_configs.db', '.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db', '.config/active_config', '.config/logs/2026.02.06/14.31.44.938153.log', '.config/logs/2026.02.06/14.30.32.592228.log', '.config/logs/2026.02.06/14.31.28.771044.log', '.config/logs/2026.02.06/14.31.35.535753.log', '.config/logs/2026.02.06/14.31.45.734270.log', '.config/logs/2026.02.06/14.31.19.332851.log', '.config/configurations/config_default', 'results/exp4_ablations.json', 'results/exp1_layer_norms.json', 'results/exp3_comparison.json', 'results/exp5_2d_sweep.json', 'results/exp2_hgf_inflation.json', 'results/figs/pareto_fid_nhfp.png', 'results/figs/sample_comparison.png', 'results/figs/precision_recall.png', 'results/figs/pareto_fid_nhfp.pdf', 'results/figs/precision_recall.pdf', 'results_imagenet256/layer_sem4.0_tex1.0.p

In [ ]:
#Time: Appx 15+ hours for entire run.
#Expect interruptions from runtime disconnects

import math
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.models as tv_models

# Ended up running 500k steps for the final numbers
TRAIN_STEPS = 500_000
CLASSIFIER_EPOCHS = 15
N_EVAL_SAMPLES = 5000
SAMPLE_STEPS = 100
SAMPLE_BATCH = 1000
BATCH_SIZE = 128
LR_DIFFUSION = 2e-4
LR_CLASSIFIER = 1e-3

CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints"
RESULTS_DIR = "/content/results"
DATA_ROOT = "/content/data"

# guidance scale values used across experiments
S_COMPARISON = [1.0, 2.0, 3.0, 5.0]
S_ABLATION = [2.0, 5.0, 7.5]
S_SEM_GRID = [2.0, 3.0, 5.0, 7.5]
S_TEX_GRID = [0.5, 1.0, 1.5, 2.0]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
T_MAX = 1000
NULL_CLASS = 10  # extra embedding index used as the "unconditional" token for CFG
P_DROP = 0.1    # probability of dropping the class label during training
CIFAR_CLASSES = 10

print(f"using device: {DEVICE}")


# cosine noise schedule from Nichol paper, better than
# the linear schedule from the original DDPM paper for CIFAR
def make_cosine_schedule(T, device="cpu"):
    s = 0.008  # small offset to avoid singularities near t=0
    steps = torch.arange(0, T + 1, dtype=torch.float64, device=device)
    f = torch.cos((steps / T + s) / (1.0 + s) * math.pi / 2.0) ** 2
    alpha_bar = (f / f[0]).clamp(1e-5, 1.0)
    betas = (1.0 - alpha_bar[1:] / alpha_bar[:-1]).clamp(0.0, 0.999)
    return {"alpha_bar": alpha_bar[1:].float(), "betas": betas.float(), "T": T}


# deterministic DDIM sampler (eta=0) from Song paper
# much faster than ancestral sampling and gives cleaner results
def ddim_step(xt, t_curr, t_prev, eps_pred, schedule):
    ab = schedule["alpha_bar"].to(xt.device)
    ab_t = ab[t_curr - 1]
    ab_prev = ab[t_prev - 1] if t_prev >= 1 else torch.tensor(1.0, device=xt.device)
    x0_hat = ((xt - (1.0 - ab_t).sqrt() * eps_pred) / ab_t.sqrt()).clamp(-1.0, 1.0)
    return ab_prev.sqrt() * x0_hat + (1.0 - ab_prev).sqrt() * eps_pred


def get_sample_timesteps(T, n_steps):
    step_size = max(T // n_steps, 1)
    steps = sorted(range(step_size, T + 1, step_size), reverse=True)
    return steps[:n_steps]


# helper to pick the largest valid number of groups for GroupNorm
def get_num_groups(channels):
    for g in [32, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g


# standard sinusoidal timestep embedding (same as in Attention is All You Need)
def sinusoidal_emb(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    angles = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([angles.sin(), angles.cos()], dim=-1)


# Adaptive Group Norm, conditions the normalization on the timestep+class embedding.
# Initializing the projection to zero
class AdaGN(nn.Module):
    def __init__(self, channels, cond_dim):
        super().__init__()
        self.gn = nn.GroupNorm(get_num_groups(channels), channels, affine=False)
        self.proj = nn.Linear(cond_dim, 2 * channels)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x, cond):
        ss = self.proj(cond).unsqueeze(-1).unsqueeze(-1)
        scale, shift = ss.chunk(2, dim=1)
        return self.gn(x) * (1.0 + scale) + shift


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, drop=0.1):
        super().__init__()
        self.norm1 = AdaGN(in_ch, cond_dim)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = AdaGN(out_ch, cond_dim)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.drop = nn.Dropout(drop)
        self.act = nn.SiLU()
        # 1x1 conv to match dimensions if needed, otherwise identity
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond):
        h = self.act(self.norm1(x, cond))
        h = self.conv1(h)
        h = self.drop(h)
        h = self.act(self.norm2(h, cond))
        h = self.conv2(h)
        return h + self.skip(x)


class Attn(nn.Module):
    def __init__(self, channels, heads=4):
        super().__init__()
        assert channels % heads == 0
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)
        self.qkv = nn.Conv1d(channels, 3 * channels, kernel_size=1, bias=False)
        self.out_proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.heads = heads
        self.head_dim = channels // heads
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).view(B, C, H * W)
        q, k, v = self.qkv(h).chunk(3, dim=1)
        # split into heads: (B, heads, seq_len, head_dim)
        q = q.view(B, self.heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        k = k.view(B, self.heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        v = v.view(B, self.heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        w = torch.softmax(q @ k.transpose(-2, -1) * self.scale, dim=-1)
        out = (w @ v).permute(0, 1, 3, 2).reshape(B, C, H * W)
        return x + self.out_proj(out).view(B, C, H, W)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, use_attn=False, drop=0.1):
        super().__init__()
        self.res1 = ResBlock(in_ch, out_ch, cond_dim, drop)
        self.res2 = ResBlock(out_ch, out_ch, cond_dim, drop)
        self.attn = Attn(out_ch) if use_attn else nn.Identity()
        self.downsample = nn.Conv2d(out_ch, out_ch, kernel_size=4, stride=2, padding=1)

    def forward(self, x, cond):
        x = self.res1(x, cond)
        x = self.res2(x, cond)
        x = self.attn(x)
        skip = x  # save before downsampling for the skip connection
        x = self.downsample(x)
        return x, skip


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, cond_dim, use_attn=False, drop=0.1):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_ch, in_ch, kernel_size=4, stride=2, padding=1)
        self.res1 = ResBlock(in_ch + skip_ch, out_ch, cond_dim, drop)
        self.res2 = ResBlock(out_ch, out_ch, cond_dim, drop)
        self.attn = Attn(out_ch) if use_attn else nn.Identity()

    def forward(self, x, skip, cond):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        x = self.res1(x, cond)
        x = self.res2(x, cond)
        return self.attn(x)


# UNet backbone: 32->16->8->4 in the encoder, then back up with skip connections.
# Attention only at the lower resolutions (8x8 and 4x4) to keep it tractable.
class UNet(nn.Module):
    def __init__(self, nc=3, base=64, mults=(1, 2, 4, 8), nclass=10, null=10, drop=0.1):
        super().__init__()
        self.null = null
        dims = [base * m for m in mults]
        cond_dim = dims[0] * 4  # project conditioning up to a wider space

        self.time_emb = nn.Sequential(
            nn.Linear(dims[0], cond_dim),
            nn.SiLU(),
            nn.Linear(cond_dim, cond_dim)
        )
        self.class_emb = nn.Embedding(nclass + 1, cond_dim)  # +1 for null class
        self.in_conv = nn.Conv2d(nc, dims[0], kernel_size=7, padding=3)

        self.encoder = nn.ModuleList([
            DownBlock(dims[0], dims[1], cond_dim, use_attn=False, drop=drop),
            DownBlock(dims[1], dims[2], cond_dim, use_attn=True,  drop=drop),
            DownBlock(dims[2], dims[3], cond_dim, use_attn=True,  drop=drop),
        ])

        # bottleneck
        self.mid1 = ResBlock(dims[3], dims[3], cond_dim, drop)
        self.mid_attn = Attn(dims[3])
        self.mid2 = ResBlock(dims[3], dims[3], cond_dim, drop)

        self.decoder = nn.ModuleList([
            UpBlock(dims[3], dims[3], dims[2], cond_dim, use_attn=True,  drop=drop),
            UpBlock(dims[2], dims[2], dims[1], cond_dim, use_attn=True,  drop=drop),
            UpBlock(dims[1], dims[1], dims[0], cond_dim, use_attn=False, drop=drop),
        ])

        self.out_norm = nn.GroupNorm(get_num_groups(dims[0]), dims[0])
        self.out_conv = nn.Conv2d(dims[0], nc, kernel_size=3, padding=1)
        self.act = nn.SiLU()

    def get_cond_emb(self, t, lbl):
        t_emb = self.time_emb(sinusoidal_emb(t, self.time_emb[0].in_features))
        return t_emb + self.class_emb(lbl)

    def encode(self, x, emb):
        h = self.in_conv(x)
        skips = []
        for block in self.encoder:
            h, sk = block(h, emb)
            skips.append(sk)
        h = self.mid1(h, emb)
        h = self.mid_attn(h)
        h = self.mid2(h, emb)
        return h, skips

    def decode(self, h, skips, emb):
        for up_block, skip in zip(self.decoder, reversed(skips)):
            h = up_block(h, skip, emb)
        return self.out_conv(self.act(self.out_norm(h)))

    def forward(self, x, t, lbl):
        emb = self.get_cond_emb(t, lbl)
        h, skips = self.encode(x, emb)
        return self.decode(h, skips, emb)


# simple config object to pass guidance parameters around without a huge argument list
class GuidanceConfig:
    def __init__(self, method="cfg", s=1.5, s_sem=3.0, s_tex=1.5, lam_cfgpp=1.0, label=""):
        self.method = method
        self.s = s
        self.s_sem = s_sem
        self.s_tex = s_tex
        self.lam_cfgpp = lam_cfgpp
        self.label = label


# cache frequency masks so we're not rebuilding them every diffusion step
_mask_cache = {}

def get_freq_masks(H, W, f_cut, device, transition=1.5):
    key = (H, W, f_cut, device)
    if key not in _mask_cache:
        fy = torch.fft.fftfreq(H, d=1.0/H, device=device)
        fx = torch.fft.fftfreq(W, d=1.0/W, device=device)
        gy, gx = torch.meshgrid(fy, fx, indexing="ij")
        freq = (gy**2 + gx**2).sqrt()
        # soft sigmoid boundary instead of a hard cutoff -- less ringing
        low_mask = torch.sigmoid((f_cut - freq) / transition)
        _mask_cache[key] = (low_mask, 1.0 - low_mask)
    return _mask_cache[key]


# LayerCFG: split the guidance signal into low-freq (semantic) and 
# high-freq (texture) components and apply different scales to each.
def layer_cfg_step(model, xt, t, lbl, null_lbl, s_sem, s_tex, f_cut=8):
    eps_cond   = model(xt, t, lbl).float()
    eps_uncond = model(xt, t, null_lbl).float()
    delta = eps_cond - eps_uncond

    B, C, H, W = delta.shape
    low_mask, high_mask = get_freq_masks(H, W, f_cut, delta.device)

    delta_fft = torch.fft.fft2(delta)
    delta_low = torch.fft.ifft2(delta_fft * low_mask).real   # semantic component
    delta_high = torch.fft.ifft2(delta_fft * high_mask).real  # texture component

    return eps_uncond + s_sem * delta_low + s_tex * delta_high


@torch.no_grad()
def _sample_chunk(model, cfg, cls, schedule, n, device):
    timesteps = get_sample_timesteps(T_MAX, SAMPLE_STEPS)
    ab = schedule["alpha_bar"].to(device)
    xt = torch.randn(n, 3, 32, 32, device=device)
    lbl = torch.full((n,), cls,        dtype=torch.long, device=device)
    null = torch.full((n,), model.null, dtype=torch.long, device=device)
    use_amp = (device == "cuda")

    for i, t_val in enumerate(timesteps):
        t_next = timesteps[i + 1] if i < len(timesteps) - 1 else 0
        t_batch = torch.full((n,), t_val, dtype=torch.long, device=device)

        with torch.cuda.amp.autocast(enabled=use_amp):
            if cfg.method == "cfg":
                eps_c = model(xt, t_batch, lbl)
                eps_u = model(xt, t_batch, null)
                ep = eps_u + cfg.s * (eps_c - eps_u)

            elif cfg.method == "rescale":
                # variance-preserving rescale from Lin et al. 2023
                eps_c = model(xt, t_batch, lbl)
                eps_u = model(xt, t_batch, null)
                guided = eps_u + cfg.s * (eps_c - eps_u)
                std_c = eps_c.view(n, -1).std(1).clamp(1e-8).view(n, 1, 1, 1)
                std_g = guided.view(n, -1).std(1).clamp(1e-8).view(n, 1, 1, 1)
                ep = guided * (std_c / std_g)

            elif cfg.method == "cfgpp":
                # CFG++ from Chung et al. 2024, operates in x0 space
                eps_c = model(xt, t_batch, lbl).float()
                eps_u = model(xt, t_batch, null).float()
                ab_t = ab[t_val - 1]
                guided = eps_u + cfg.s * (eps_c - eps_u)
                x0_u = (xt - (1 - ab_t).sqrt() * eps_u)    / ab_t.sqrt()
                x0_g = (xt - (1 - ab_t).sqrt() * guided)   / ab_t.sqrt()
                x0_c = (x0_u + cfg.lam_cfgpp * (x0_g - x0_u)).clamp(-1, 1)
                ep = (xt - ab_t.sqrt() * x0_c) / (1 - ab_t).sqrt()

            elif cfg.method == "layer_cfg":
                ep = layer_cfg_step(model, xt, t_batch, lbl, null, cfg.s_sem, cfg.s_tex)

        xt = ddim_step(xt, t_val, t_next, ep.float(), schedule)

    return xt.clamp(-1, 1).cpu()


def sample_images(model, cfg, cls, schedule, n, device):
    model.eval()
    chunks = []
    remaining = n
    while remaining > 0:
        bs = min(SAMPLE_BATCH, remaining)
        chunks.append(_sample_chunk(model, cfg, cls, schedule, bs, device))
        remaining -= bs
    return torch.cat(chunks, dim=0)


# normalized high-frequency power, ratio of HF energy in generated vs real images.
# > 1.0 means the model is producing too much texture / is over-sharpened.
def compute_nhfp(gen_imgs, ref_imgs, f_cut=8):
    def high_freq_power(imgs):
        imgs_01 = (imgs.float() + 1) / 2
        H, W = imgs_01.shape[-2], imgs_01.shape[-1]
        fy = torch.fft.fftfreq(H, d=1.0/H)
        fx = torch.fft.fftfreq(W, d=1.0/W)
        gy, gx = torch.meshgrid(fy, fx, indexing="ij")
        hf_mask = ((gy**2 + gx**2).sqrt() > f_cut).float()
        fft = torch.fft.fft2(imgs_01)
        power = fft.abs().pow(2)
        hf_power = (power * hf_mask).sum((-2, -1))
        return hf_power / power.sum((-2, -1)).clamp(1e-8)

    gen_hfp = high_freq_power(gen_imgs.cpu()).mean().item()
    ref_hfp = high_freq_power(ref_imgs.cpu()).mean().item()
    return gen_hfp / max(ref_hfp, 1e-8)


# thin wrapper around InceptionV3 for extracting pool3 features (FID)
# or logits (IS). Resizes to 299x299 and uses ImageNet normalization.
class InceptionWrapper(nn.Module):
    def __init__(self, mode, device):
        super().__init__()
        assert mode in ("pool3", "logits")
        model = tv_models.inception_v3(weights=tv_models.Inception_V3_Weights.DEFAULT)
        model.eval()
        model.aux_logits = False
        if mode == "pool3":
            model.fc = nn.Identity()
        self.model = model.to(device)
        self.mode = mode
        self.device = device
        import torchvision.transforms as T
        self.preprocess = T.Compose([
            T.Resize((299, 299), antialias=True),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    @torch.no_grad()
    def extract(self, imgs, bs=64):
        results = []
        for i in range(0, len(imgs), bs):
            batch = ((imgs[i:i+bs].float() + 1) / 2).to(self.device)
            out = self.model(self.preprocess(batch))
            if isinstance(out, tv_models.inception.InceptionOutputs):
                out = out.logits
            results.append(out.cpu().numpy())
        return np.concatenate(results, axis=0)


def compute_fid(gen_feats, real_feats):
    from scipy.linalg import sqrtm
    mu_gen = gen_feats.mean(0)
    mu_real = real_feats.mean(0)
    cov_gen = np.cov(gen_feats,  rowvar=False)
    cov_real = np.cov(real_feats, rowvar=False)
    diff = mu_real - mu_gen
    cov_mean, _ = sqrtm(cov_real @ cov_gen, disp=False)
    if np.iscomplexobj(cov_mean):
        cov_mean = cov_mean.real  # numerical issues sometimes produce tiny imaginary parts
    return float(diff @ diff + np.trace(cov_real + cov_gen - 2 * cov_mean))


def inception_score(logits, splits=5):
    logits = logits - logits.max(1, keepdims=True)
    p = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
    N = len(p)
    scores = []
    for k in range(splits):
        part = p[k * (N // splits):(k + 1) * (N // splits)]
        py = part.mean(0, keepdims=True)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        scores.append(np.exp(kl.sum(1).mean()))
    return float(np.mean(scores)), float(np.std(scores))


def classifier_alignment(gen_imgs, target_lbls, clf, device, bs=256):
    clf.eval()
    correct = 0
    for i in range(0, len(gen_imgs), bs):
        imgs = gen_imgs[i:i+bs].to(device)
        preds = clf(imgs).argmax(1)
        correct += (preds == target_lbls[i:i+bs].to(device)).sum().item()
    return correct / len(gen_imgs)


def precision_recall(gen_feats, real_feats, k=3):
    from sklearn.neighbors import NearestNeighbors

    def get_knn_radii(X):
        nn_model = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(X)
        dists, _ = nn_model.kneighbors(X)
        return dists[:, -1]

    real_radii = get_knn_radii(real_feats)
    gen_radii  = get_knn_radii(gen_feats)

    def fraction_covered(queries, centers, radii, chunk=50):
        total = 0
        for i in range(0, len(queries), chunk):
            q = queries[i:i+chunk]
            dists = np.linalg.norm(q[:, None] - centers[None], axis=-1)
            total += (dists <= radii[None]).any(1).sum()
        return total / len(queries)

    prec = fraction_covered(gen_feats, real_feats, real_radii)
    rec  = fraction_covered(real_feats, gen_feats, gen_radii)
    return prec, rec


def get_loaders(bs=128, root=DATA_ROOT):
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    val_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    train_loader = DataLoader(
        datasets.CIFAR10(root, train=True, download=True, transform=train_tf),
        batch_size=bs, shuffle=True, num_workers=8,
        pin_memory=True, drop_last=True, persistent_workers=True
    )
    val_loader = DataLoader(
        datasets.CIFAR10(root, train=False, download=False, transform=val_tf),
        batch_size=bs, shuffle=False, num_workers=8,
        pin_memory=True, persistent_workers=True
    )
    return train_loader, val_loader


def real_images_balanced(ds, n=1000):
    # pull an equal number of images from each class so metrics aren't biased
    per_class = n // CIFAR_CLASSES
    counts = {c: 0 for c in range(CIFAR_CLASSES)}
    tensors = []
    for img, lbl in ds:
        c = int(lbl)
        if counts[c] < per_class:
            tensors.append(img)
            counts[c] += 1
        if sum(counts.values()) >= per_class * CIFAR_CLASSES:
            break
    return torch.stack(tensors)


def train_diffusion(resume=True):
    print("\ntraining diffusion model...")
    schedule  = make_cosine_schedule(T_MAX, "cpu")
    ab = schedule["alpha_bar"].to(DEVICE)
    model = UNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_DIFFUSION)
    start_step = 0

    ckpt_path = f"{CHECKPOINT_DIR}/diffusion_latest.pt"
    if resume and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        start_step = ckpt["step"]
        print(f"  resumed from step {start_step}")

    if start_step >= TRAIN_STEPS:
        print("  already done training")
        return model, schedule

    train_loader, _ = get_loaders(BATCH_SIZE)
    data_iter = iter(train_loader)
    model.train()

    use_amp = (DEVICE == "cuda")
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    pbar = tqdm(range(start_step, TRAIN_STEPS), desc="diffusion", initial=start_step, total=TRAIN_STEPS)
    for step in pbar:
        try:
            x0, lbl = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            x0, lbl  = next(data_iter)

        x0  = x0.to(DEVICE)
        lbl = lbl.to(DEVICE)
        B   = x0.shape[0]

        # sample random timestep and add noise
        t    = torch.randint(1, T_MAX + 1, (B,), device=DEVICE)
        ab_t = ab[t - 1].view(B, 1, 1, 1)
        noise = torch.randn_like(x0)
        xt    = ab_t.sqrt() * x0 + (1 - ab_t).sqrt() * noise

        # randomly replace labels with the null token for CFG training
        drop_mask = torch.rand(B, device=DEVICE) < P_DROP
        lbl_input = lbl.masked_fill(drop_mask, NULL_CLASS)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = F.mse_loss(model(xt, t, lbl_input), noise)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if (step + 1) % 500 == 0:
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        if (step + 1) % 10_000 == 0:
            ckpt_data = {"step": step + 1, "model": model.state_dict(), "optimizer": optimizer.state_dict()}
            torch.save(ckpt_data, ckpt_path)
            torch.save(ckpt_data, f"{CHECKPOINT_DIR}/diffusion_step{step+1:07d}.pt")

    torch.save({"step": TRAIN_STEPS, "model": model.state_dict(), "optimizer": optimizer.state_dict()}, ckpt_path)
    print("  done training diffusion")
    return model, schedule


# ResNet-18 adapted for 32x32 CIFAR images, remove the big stride-2 conv
# and max pool at the front since the images are already small
class CIFARClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = tv_models.resnet18(weights=None)
        backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        backbone.maxpool = nn.Identity()
        backbone.fc = nn.Linear(512, 10)
        self.net = backbone

    def forward(self, x):
        # model expects [0,1] but our images are in [-1,1]
        return self.net((x + 1.0) / 2.0)


def train_classifier(resume=True):
    print("\ntraining classifier...")
    clf = CIFARClassifier().to(DEVICE)
    optimizer = torch.optim.AdamW(clf.parameters(), lr=LR_CLASSIFIER, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, CLASSIFIER_EPOCHS)

    best_path = f"{CHECKPOINT_DIR}/classifier_best.pt"
    if resume and os.path.exists(best_path):
        clf.load_state_dict(torch.load(best_path, map_location=DEVICE))
        print("  loaded existing classifier")
        return clf

    train_loader, val_loader = get_loaders(256)
    best_acc = 0.0

    for epoch in range(1, CLASSIFIER_EPOCHS + 1):
        clf.train()
        for x, y in tqdm(train_loader, desc=f"  epoch {epoch}/{CLASSIFIER_EPOCHS}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            F.cross_entropy(clf(x), y).backward()
            optimizer.step()
        scheduler.step()

        clf.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                correct += (clf(x).argmax(1) == y).sum().item()
                total += y.shape[0]
        acc = correct / total
        print(f"  epoch {epoch}: val acc = {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            torch.save(clf.state_dict(), best_path)

    print(f"  best val acc: {best_acc:.4f}")
    return clf

@torch.no_grad()
def eval_config(model, cfg, schedule, pool_ext, logit_ext, clf, real_pool, real_imgs, n):
    samples_per_class = n // CIFAR_CLASSES
    all_imgs = []
    all_lbls = []

    for cls in range(CIFAR_CLASSES):
        imgs = sample_images(model, cfg, cls, schedule, samples_per_class, DEVICE)
        all_imgs.append(imgs)
        all_lbls.extend([cls] * samples_per_class)

    all_imgs = torch.cat(all_imgs, dim=0)
    all_lbls = torch.tensor(all_lbls)

    gen_pool   = pool_ext.extract(all_imgs)
    gen_logits = logit_ext.extract(all_imgs)

    fid_val = compute_fid(gen_pool, real_pool)
    is_mean, is_std  = inception_score(gen_logits)
    nhfp_val = compute_nhfp(all_imgs, real_imgs.cpu())
    prec, rec = precision_recall(gen_pool, real_pool)

    # per-class classifier alignment averaged at the end
    ca_per_class = []
    for cls in range(CIFAR_CLASSES):
        idx = (all_lbls == cls).nonzero(as_tuple=True)[0]
        imgs = all_imgs[idx].to(DEVICE)
        target = torch.full((len(imgs),), cls, dtype=torch.long, device=DEVICE)
        ca_per_class.append(classifier_alignment(imgs, target, clf, DEVICE))
    ca = float(np.mean(ca_per_class))

    return {
        "label":     cfg.label,
        "s_sem":     cfg.s_sem,
        "s_tex":     cfg.s_tex,
        "fid":       round(fid_val, 2),
        "is_mean":   round(is_mean, 3),
        "is_std":    round(is_std, 3),
        "ca":        round(ca, 4),
        "nhfp":      round(nhfp_val, 4),
        "precision": round(prec, 4),
        "recall":    round(rec, 4)
    }


def exp3_method_comparison(model, schedule, pool_ext, logit_ext, clf, rp, ri):
    print("\n--- Exp 3: CFG baseline ---")
    results = []
    for s in S_COMPARISON:
        cfg = GuidanceConfig(method="cfg", s=s, s_sem=s, s_tex=s, label=f"CFG s={s}")
        print(f"  running {cfg.label}...", flush=True)
        r = eval_config(model, cfg, schedule, pool_ext, logit_ext, clf, rp, ri, N_EVAL_SAMPLES)
        results.append(r)
        print(f"    FID={r['fid']}  IS={r['is_mean']}  CA={r['ca']}  NHFP={r['nhfp']}")
    return results


def exp5_2d_sweep(model, schedule, pool_ext, logit_ext, clf, rp, ri):
    print("\n--- Exp 5: 2D sweep ---")
    results = []
    for ss in S_SEM_GRID:
        for st in S_TEX_GRID:
            lbl = f"Split s_sem={ss} s_tex={st}"
            cfg = GuidanceConfig(method="layer_cfg", s_sem=ss, s_tex=st, label=lbl)
            print(f"  running {lbl}...", flush=True)
            r = eval_config(model, cfg, schedule, pool_ext, logit_ext, clf, rp, ri, N_EVAL_SAMPLES)
            results.append(r)
            print(f"    FID={r['fid']}  CA={r['ca']}  NHFP={r['nhfp']}")
    return results


def print_results_table(results, title):
    print(f"\n{title}")
    print("-" * 86)
    print(f"  {'Method':<28}  {'s_sem':<6}  {'s_tex':<6}  {'FID':<8}  {'IS':<7}  {'CA':<7}  {'NHFP':<7}  {'Prec':<7}  {'Rec':<7}")
    print("-" * 86)
    for r in results:
        s_sem = r.get("s_sem", "-")
        s_tex = r.get("s_tex", "-")
        print(f"  {r['label']:<28}  {str(s_sem):<6}  {str(s_tex):<6}  {r['fid']:<8}  {r['is_mean']:<7}  {r['ca']:<7}  {r['nhfp']:<7}  {r['precision']:<7}  {r['recall']:<7}")


def save_json(name, obj):
    path = f"{RESULTS_DIR}/{name}.json"
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"  saved {path}")


def _save_visuals(model, schedule, e3, e5):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    os.makedirs(f"{RESULTS_DIR}/figs", exist_ok=True)

    CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                       "dog", "frog", "horse", "ship", "truck"]

    # pick the best LayerCFG config by FID (filtered to FID < 25 to avoid degenerate cases)
    best_layer = min((r for r in e5 if r["fid"] < 25), key=lambda r: r["fid"])
    cfg_baseline = GuidanceConfig(method="cfg", s=3.0, label="CFG s=3")
    cfg_layer = GuidanceConfig(
        method="layer_cfg",
        s_sem=best_layer["s_sem"],
        s_tex=best_layer["s_tex"],
        label=best_layer["label"]
    )

    cfg_imgs = []
    layer_imgs = []
    for cls in range(CIFAR_CLASSES):
        img_cfg = sample_images(model, cfg_baseline, cls, schedule, 1, DEVICE)
        img_layer = sample_images(model, cfg_layer,    cls, schedule, 1, DEVICE)
        cfg_imgs.append(((img_cfg[0].clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy())
        layer_imgs.append(((img_layer[0].clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy())

    fig, axes = plt.subplots(2, 10, figsize=(20, 4.5))
    for col, name in enumerate(CIFAR10_CLASSES):
        axes[0, col].imshow(cfg_imgs[col])
        axes[0, col].axis("off")
        axes[0, col].set_title(name, fontsize=8)
        axes[1, col].imshow(layer_imgs[col])
        axes[1, col].axis("off")
    axes[0, 0].set_ylabel("CFG s=3\n(NHFP=1.19)", fontsize=9, labelpad=6)
    axes[1, 0].set_ylabel(f"LayerCFG\n(NHFP={best_layer['nhfp']})", fontsize=9, labelpad=6)
    plt.suptitle("CFG vs LayerCFG — one sample per class", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/figs/sample_comparison.png", dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  saved sample_comparison.png  (best: {best_layer['label']})")

    # Pareto frontier: FID vs NHFP
    fig, ax = plt.subplots(figsize=(7, 5))
    cfg_pts = [(r["nhfp"], r["fid"], f"s={r['s_sem']}") for r in e3 if r["label"].startswith("CFG s=")]
    ax.plot([p[0] for p in cfg_pts], [p[1] for p in cfg_pts],
            "o-", color="steelblue", label="CFG", zorder=3)
    for nhfp_val, fid_val, lbl in cfg_pts:
        ax.annotate(lbl, (nhfp_val, fid_val), textcoords="offset points",
                    xytext=(4, 4), fontsize=7, color="steelblue")

    colors = {2.0: "#e377c2", 3.0: "#d62728", 5.0: "#8c564b"}
    for s_sem in [2.0, 3.0, 5.0]:
        pts = sorted(
            [(r["nhfp"], r["fid"]) for r in e5
             if abs(r.get("s_sem", 0) - s_sem) < 0.1 and r["fid"] < 30],
            key=lambda x: x[0]
        )
        if pts:
            ax.plot([p[0] for p in pts], [p[1] for p in pts], "o-",
                    color=colors[s_sem], label=f"LayerCFG s_sem={s_sem}", linewidth=1.5)

    ax.set_xlabel("NHFP (lower = less texture artifact)", fontsize=11)
    ax.set_ylabel("FID (lower = better)", fontsize=11)
    ax.set_title("FID vs NHFP Pareto Frontier", fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.axvline(1.0, color="gray", linestyle=":", linewidth=1)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/figs/pareto_fid_nhfp.pdf", bbox_inches="tight")
    plt.savefig(f"{RESULTS_DIR}/figs/pareto_fid_nhfp.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  saved pareto_fid_nhfp.pdf/.png")

    fig, ax = plt.subplots(figsize=(6, 5))
    cfg_pr = [(r["recall"], r["precision"], f"s={r['s_sem']}") for r in e3]
    ax.plot([p[0] for p in cfg_pr], [p[1] for p in cfg_pr],
            "o-", color="steelblue", label="CFG", zorder=3)
    for rec_val, prec_val, lbl in cfg_pr:
        ax.annotate(lbl, (rec_val, prec_val), textcoords="offset points",
                    xytext=(4, 4), fontsize=7, color="steelblue")
    for s_sem, color in [(2.0, "#e377c2"), (3.0, "#d62728"), (5.0, "#8c564b")]:
        pts = [(r["recall"], r["precision"]) for r in e5
               if abs(r.get("s_sem", 0) - s_sem) < 0.1 and r["fid"] < 30]
        if pts:
            ax.scatter([p[0] for p in pts], [p[1] for p in pts],
                       color=color, label=f"LayerCFG s_sem={s_sem}", zorder=4, s=40)
    ax.set_xlabel("Recall (diversity)", fontsize=11)
    ax.set_ylabel("Precision (fidelity)", fontsize=11)
    ax.set_title("Precision vs Recall", fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/figs/precision_recall.pdf", bbox_inches="tight")
    plt.savefig(f"{RESULTS_DIR}/figs/precision_recall.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  saved precision_recall.pdf/.png")


def main():
    t0 = time.time()
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(RESULTS_DIR, exist_ok=True)

    model, schedule = train_diffusion(resume=True)
    clf = train_classifier(resume=True)
    model.eval()

    print("\nloading data and inception model...")
    _, val_loader = get_loaders(128)
    val_ds = datasets.CIFAR10(DATA_ROOT, train=False, download=False, transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]))
    real_imgs = real_images_balanced(val_ds, N_EVAL_SAMPLES)
    pool_ext  = InceptionWrapper("pool3",  DEVICE)
    logit_ext = InceptionWrapper("logits", DEVICE)
    real_pool = pool_ext.extract(real_imgs.cpu())

    # load from disk if we've already run this (saves a lot of time)
    e3_path = f"{RESULTS_DIR}/exp3_comparison.json"
    if os.path.exists(e3_path):
        with open(e3_path) as f:
            e3 = [r for r in json.load(f) if r["label"].startswith("CFG s=")]
        print("  loaded exp 3 from disk")
    else:
        e3 = exp3_method_comparison(model, schedule, pool_ext, logit_ext, clf, real_pool, real_imgs)
        save_json("exp3_comparison", e3)
    print_results_table(e3, "CFG Baseline")

    e5_path = f"{RESULTS_DIR}/exp5_2d_sweep.json"
    if os.path.exists(e5_path):
        with open(e5_path) as f:
            e5 = json.load(f)
        print("  loaded exp 5 from disk")
    else:
        e5 = exp5_2d_sweep(model, schedule, pool_ext, logit_ext, clf, real_pool, real_imgs)
        save_json("exp5_2d_sweep", e5)
    print_results_table(e5, "LayerCFG 2D Sweep")

    best_cfg  = min(e3, key=lambda r: r["fid"])
    best_fid  = min(e5, key=lambda r: r["fid"])
    best_nhfp = min(e5, key=lambda r: abs(r["nhfp"] - 1.0))

    print("\n--- summary ---")
    print(f"  best CFG:           {best_cfg['label']}  FID={best_cfg['fid']}  NHFP={best_cfg['nhfp']}")
    print(f"  best LayerCFG FID:  {best_fid['label']}  FID={best_fid['fid']}  NHFP={best_fid['nhfp']}")
    print(f"  best LayerCFG NHFP: {best_nhfp['label']}  FID={best_nhfp['fid']}  NHFP={best_nhfp['nhfp']}")

    _save_visuals(model, schedule, e3, e5)

    elapsed = (time.time() - t0) / 60
    print(f"\n  total time: {elapsed:.1f} min")
    print(f"  results in {RESULTS_DIR}/")


if __name__ == "__main__":
    main()


Device: cpu

TRAINING: Diffusion UNet
  Resumed from step 500000
  Already trained to target steps.

TRAINING: CIFAR-10 Classifier
  Loaded existing classifier.

Loading data and Inception extractors …
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 171MB/s] 


  Loaded Exp 3 from disk (CFG only).

  EXP 3 — CFG Baseline
  Method                        s_sem   s_tex   FID       IS       CA       NHFP     Prec     Recall 
  --------------------------------------------------------------------------------------
  CFG s=1.0                     1.0     1.0     22.33     4.784    0.8312   1.0892   0.6942   0.5354 
  CFG s=2.0                     2.0     2.0     15.49     4.846    0.9664   1.0835   0.762    0.5116 
  CFG s=3.0                     3.0     3.0     15.98     4.801    0.9862   1.1875   0.7804   0.4744 
  CFG s=5.0                     5.0     5.0     21.24     4.88     0.965    1.4035   0.695    0.439  
  Loaded Exp 5 from disk.

  EXP 5 — LayerCFG 2-D Sweep (s_sem × s_tex)
  Method                        s_sem   s_tex   FID       IS       CA       NHFP     Prec     Recall 
  --------------------------------------------------------------------------------------
  Split s_sem=2.0 s_tex=0.5     2.0     0.5     17.65     4.783    0.9438   1

In [ ]:
!pip install diffusers accelerate transformers -q


In [ ]:
# ImageNet 256x256 experiment - DiT-XL/2
# NOTE: first run downloads ~2GB from huggingface, takes a while

import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
import torchvision.models as tv_models
import torchvision.transforms as T
# import matplotlib.pyplot as plt  # causes backend issues if imported globally

# main hyperparams
N_PER_CLASS = 8       # how many images to generate per class
SAMPLE_STEPS = 50     # diffusion steps (50 is fine, 250 would be overkill)
CFG_SCALE = 4.0       # standard CFG scale we're comparing against

# 2D sweep over semantic vs texture guidance -- wanted more values but it was taking forever
S_SEM_GRID = [1.5, 2.0, 2.5, 3.0, 4.0]
S_TEX_GRID = [0.1, 0.25, 0.5, 0.75, 1.0]

# latent space is 32x32 so f_cut=8 gives us the same relative cutoff as on CIFAR
F_CUT = 8
TRANSITION = 1.5  # sigmoid sharpness for the frequency split

RESULTS_DIR = "/content/results_imagenet256"
CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints_imagenet"

# picked 10 visually distinct classes so the comparison grid is easy to read
SAMPLE_CLASSES = [
    207,   # golden retriever
    388,   # giant panda
    417,   # balloon
    472,   # pineapple
    84,    # peacock
    130,   # flamingo
    980,   # volcano
    817,   # sports car
    850,   # teddy bear
    717,   # pickup truck
]

IMAGENET_NAMES = {
    207: "golden retriever",
    388: "giant panda",
    417: "balloon",
    472: "pineapple",
    84:  "peacock",
    130: "flamingo",
    980: "volcano",
    817: "sports car",
    850: "teddy bear",
    717: "pickup truck",
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


def load_dit(device=DEVICE):
    from diffusers import DiTPipeline, DDIMScheduler
    import diffusers
    diffusers.utils.logging.set_verbosity_error()

    print("Loading DiT-XL/2-256 …", flush=True)
    pipe = DiTPipeline.from_pretrained(
        "facebook/DiT-XL-2-256",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        token=False,
    ).to(device)

    # swap default scheduler for DDIM so sampling is deterministic
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    pipe.scheduler.set_timesteps(SAMPLE_STEPS)

    transformer = pipe.transformer.eval()
    scheduler = pipe.scheduler
    vae = pipe.vae.eval()

    # DiT uses class index 1000 as the null/unconditional label
    config = dict(transformer.config)
    null_class = config.get("num_class_embeds", 1000)
    latent_res  = config.get("sample_size", 32)
    print(f"  Loaded. null_class={null_class}, latent_size={latent_res}x{latent_res}")
    return transformer, scheduler, vae, null_class


# cache these so we don't recompute the masks every single diffusion step
_FREQ_MASK_CACHE = {}

def _get_freq_masks(H, W, f_cut, device, transition=TRANSITION):
    key = (H, W, f_cut, str(device))
    if key not in _FREQ_MASK_CACHE:
        fy = torch.fft.fftfreq(H, d=1.0/H, device=device)
        fx = torch.fft.fftfreq(W, d=1.0/W, device=device)
        grid_y, grid_x = torch.meshgrid(fy, fx, indexing="ij")
        radius = (grid_y**2 + grid_x**2).sqrt()

        # soft split via sigmoid, hard threshold caused ringing artifacts
        low_pass = torch.sigmoid((f_cut - radius) / transition)
        high_pass = 1.0 - low_pass

        _FREQ_MASK_CACHE[key] = (low_pass, high_pass)
    return _FREQ_MASK_CACHE[key]


@torch.no_grad()
def _sample_cfg(transformer, scheduler, null_class, class_id, n, cfg_scale, device):
    """Standard CFG, our baseline."""
    class_labels = torch.tensor([class_id]  * n, dtype=torch.long, device=device)
    null_labels = torch.tensor([null_class] * n, dtype=torch.long, device=device)

    config = dict(transformer.config)
    H = config.get("sample_size", 32)
    W = H
    C = config.get("in_channels", 4)

    dtype = torch.float16 if device == "cuda" else torch.float32
    xt = torch.randn(n, C, H, W, device=device, dtype=dtype)

    for t in scheduler.timesteps:
        t_batch = t.expand(n).to(device)

        # DiT outputs noise + variance concatenated, we only need the noise half
        noise_cond = transformer(xt, timestep=t_batch, class_labels=class_labels).sample.chunk(2, dim=1)[0]
        noise_uncond = transformer(xt, timestep=t_batch, class_labels=null_labels).sample.chunk(2, dim=1)[0]

        guided_noise = noise_uncond + cfg_scale * (noise_cond - noise_uncond)
        xt = scheduler.step(guided_noise, t, xt).prev_sample

    return xt.float().cpu()


@torch.no_grad()
def _sample_layer_cfg(transformer, scheduler, null_class, class_id, n,
                      s_sem, s_tex, f_cut, device):
    """
    LayerCFG: split the guidance signal into low-freq (semantic) and
    high-freq (texture) parts and scale them independently.
    """
    class_labels = torch.tensor([class_id]  * n, dtype=torch.long, device=device)
    null_labels  = torch.tensor([null_class] * n, dtype=torch.long, device=device)

    config = dict(transformer.config)
    H = config.get("sample_size", 32)
    W = H
    C = config.get("in_channels", 4)

    dtype = torch.float16 if device == "cuda" else torch.float32
    xt = torch.randn(n, C, H, W, device=device, dtype=dtype)

    for t in scheduler.timesteps:
        t_batch = t.expand(n).to(device)

        noise_cond   = transformer(xt, timestep=t_batch, class_labels=class_labels).sample.chunk(2, dim=1)[0].float()
        noise_uncond = transformer(xt, timestep=t_batch, class_labels=null_labels).sample.chunk(2, dim=1)[0].float()

        delta = noise_cond - noise_uncond
        _, _, dH, dW = delta.shape
        low_mask, high_mask = _get_freq_masks(dH, dW, f_cut, device)

        # FFT the guidance signal and split into semantic vs texture components
        delta_fft = torch.fft.fft2(delta)
        delta_sem = torch.fft.ifft2(delta_fft * low_mask).real
        delta_tex = torch.fft.ifft2(delta_fft * high_mask).real

        # apply separate scales and put it back together
        guided_noise = noise_uncond + s_sem * delta_sem + s_tex * delta_tex
        xt = scheduler.step(guided_noise.to(xt.dtype), t, xt).prev_sample

    return xt.float().cpu()


def sample_all_classes(transformer, scheduler, null_class, classes, n_per_class,
                       method, cfg_scale=None, s_sem=None, s_tex=None, f_cut=F_CUT):
    all_imgs = []
    t_start  = time.time()

    for i, cls in enumerate(classes):
        if method == "cfg":
            imgs = _sample_cfg(transformer, scheduler, null_class, cls,
                               n_per_class, cfg_scale, DEVICE)
        else:
            imgs = _sample_layer_cfg(transformer, scheduler, null_class, cls,
                                     n_per_class, s_sem, s_tex, f_cut, DEVICE)

        all_imgs.append(imgs)

        done    = i + 1
        elapsed = time.time() - t_start
        eta     = elapsed / done * (len(classes) - done)
        print(f"    {done}/{len(classes)} classes  ~{eta/60:.1f} min remaining", flush=True)

    return torch.cat(all_imgs, 0)


def compute_hfp(imgs, f_cut=F_CUT):
    """High-frequency power ratio, fraction of total power above f_cut."""
    H, W = imgs.shape[-2:]
    fy = torch.fft.fftfreq(H, d=1.0/H)
    fx = torch.fft.fftfreq(W, d=1.0/W)
    grid_y, grid_x = torch.meshgrid(fy, fx, indexing="ij")
    high_freq_mask = ((grid_y**2 + grid_x**2).sqrt() > f_cut).float()

    power_spectrum = torch.fft.fft2(imgs.float()).abs().pow(2)
    high_power  = (power_spectrum * high_freq_mask).sum((-2, -1))
    total_power = power_spectrum.sum((-2, -1)).clamp(1e-8)
    return (high_power / total_power).mean().item()


class InceptionWrapper(nn.Module):
    """Thin wrapper around V3, can return pool3 features or logits."""

    def __init__(self, mode, device):
        super().__init__()
        assert mode in ("pool3", "logits")

        model = tv_models.inception_v3(weights=tv_models.Inception_V3_Weights.DEFAULT)
        model.eval()
        model.aux_logits = False
        if mode == "pool3":
            model.fc = nn.Identity()

        self.model  = model.to(device)
        self.mode   = mode
        self.device = device

        # standard ImageNet mean/std
        self.preprocess = T.Compose([
            T.Resize((299, 299), antialias=True),
            T.Normalize([.485, .456, .406], [.229, .224, .225]),
        ])

    @torch.no_grad()
    def extract(self, imgs, bs=16):
        outputs = []
        for i in range(0, len(imgs), bs):
            batch = imgs[i:i+bs].float()
            x   = ((batch + 1) / 2).to(self.device)  # [-1,1] -> [0,1]
            out = self.model(self.preprocess(x))
            if hasattr(out, "logits"):
                out = out.logits
            outputs.append(out.cpu().numpy())
        return np.concatenate(outputs, 0)


def inception_score(logits, splits=5):
    # subtract max for numerical stability before softmax
    logits = logits - logits.max(1, keepdims=True)
    probs  = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
    N = len(probs)

    scores = []
    for k in range(splits):
        part = probs[k*(N//splits):(k+1)*(N//splits)]
        marginal = part.mean(0, keepdims=True)
        kl = part * (np.log(part + 1e-10) - np.log(marginal + 1e-10))
        scores.append(np.exp(kl.sum(1).mean()))

    return float(np.mean(scores)), float(np.std(scores))


@torch.no_grad()
def decode_to_pixels(latents, vae, chunk=4):
    """ Decode latents pixels in small chunks so we don't OOM. """
    vae_dtype = next(vae.parameters()).dtype
    decoded_imgs = []
    for i in range(0, len(latents), chunk):
        batch = latents[i:i+chunk].to(DEVICE, dtype=vae_dtype)
        decoded = vae.decode(batch / 0.18215).sample
        decoded_imgs.append(decoded.float().cpu().clamp(-1, 1))
    return torch.cat(decoded_imgs, 0)


def save_comparison_grid(cfg_imgs, layer_imgs, classes, label_cfg, label_layer,
                         n_per_class, save_path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    n_cls = len(classes)
    fig, axes = plt.subplots(2, n_cls, figsize=(n_cls * 2.5, 5.5))

    def to_np(t):
        return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy().clip(0, 1)

    for col, cls in enumerate(classes):
        idx = col * n_per_class

        axes[0, col].imshow(to_np(cfg_imgs[idx]))
        axes[0, col].axis("off")
        axes[0, col].set_title(IMAGENET_NAMES.get(cls, str(cls)), fontsize=7)

        axes[1, col].imshow(to_np(layer_imgs[idx]))
        axes[1, col].axis("off")

    axes[0, 0].set_ylabel(label_cfg,   fontsize=9, labelpad=6)
    axes[1, 0].set_ylabel(label_layer, fontsize=9, labelpad=6)

    plt.suptitle("CFG vs LayerCFG — ImageNet 256×256  (one sample per class)",
                 fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {save_path}")


def save_hfp_bar_chart(results, save_path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    labels = [r["label"] for r in results]
    hfps = [r["hfp"]   for r in results]
    colors = ["steelblue" if r["method"] == "cfg" else "#d62728" for r in results]

    fig, ax = plt.subplots(figsize=(max(7, len(results) * 0.9), 4))
    ax.bar(range(len(labels)), hfps, color=colors)
    ax.axhline(hfps[0], color="steelblue", linestyle="--", linewidth=1,
               label=f"CFG baseline HFP={hfps[0]:.4f}")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("High-Frequency Power (fraction)", fontsize=10)
    ax.set_title("HFP by method  (lower = less texture artifact)", fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {save_path}")


def save_pareto_plot(results, save_path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 5))

    cfg_points = [(r["hfp"], r["is_mean"], r["label"])                    for r in results if r["method"] == "cfg"]
    layer_points = [(r["hfp"], r["is_mean"], r["label"], r["s_sem"])        for r in results if r["method"] == "layer_cfg"]

    if cfg_points:
        ax.scatter([p[0] for p in cfg_points], [p[1] for p in cfg_points],
                   color="steelblue", s=80, zorder=4, label="CFG")
        for hfp, is_mean, label in cfg_points:
            ax.annotate(label, (hfp, is_mean), textcoords="offset points",
                        xytext=(4, 4), fontsize=7, color="steelblue")

    # different color per s_sem value so we can see trends
    sem_colors = {1.5: "#aec7e8", 2.0: "#e377c2", 2.5: "#ff7f0e", 3.0: "#d62728", 4.0: "#8c564b"}
    for s_sem in S_SEM_GRID:
        pts = [(r[0], r[1]) for r in layer_points if abs(r[3] - s_sem) < 0.1]
        if pts:
            ax.scatter([p[0] for p in pts], [p[1] for p in pts],
                       color=sem_colors.get(s_sem, "gray"), s=60, zorder=3,
                       label=f"LayerCFG s_sem={s_sem}", alpha=0.8)

    ax.set_xlabel("HFP (lower = less texture artifact)", fontsize=11)
    ax.set_ylabel("Inception Score (higher = better)", fontsize=11)
    ax.set_title("HFP vs IS  (want: low HFP, high IS)", fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {save_path}")


def main():
    t0 = time.time()
    os.makedirs(RESULTS_DIR, exist_ok=True)
    os.makedirs(f"{RESULTS_DIR}/figs", exist_ok=True)

    transformer, scheduler, vae, null_class = load_dit(DEVICE)

    print("Loading Inception V3 for IS metric…")
    logit_extractor = InceptionWrapper("logits", DEVICE)

    results = []

    #standard CFG
    print(f"\n--- CFG s={CFG_SCALE} ---")
    cfg_cache_path = f"{RESULTS_DIR}/cfg_imgs.pt"
    if os.path.exists(cfg_cache_path):
        cfg_imgs = torch.load(cfg_cache_path)
        print("  Loaded from disk.")
    else:
        cfg_imgs = sample_all_classes(transformer, scheduler, null_class,
                                      SAMPLE_CLASSES, N_PER_CLASS,
                                      method="cfg", cfg_scale=CFG_SCALE)
        cfg_imgs = decode_to_pixels(cfg_imgs, vae)
        torch.save(cfg_imgs, cfg_cache_path)

    cfg_logits = logit_extractor.extract(cfg_imgs)
    cfg_is_mean, cfg_is_std = inception_score(cfg_logits)
    cfg_hfp = compute_hfp(cfg_imgs)

    cfg_result = {
        "label":   f"CFG s={CFG_SCALE}",
        "method":  "cfg",
        "hfp":     round(cfg_hfp, 5),
        "is_mean": round(cfg_is_mean, 3),
        "is_std":  round(cfg_is_std, 3),
    }
    results.append(cfg_result)
    print(f"  IS={cfg_is_mean:.3f}±{cfg_is_std:.3f}  HFP={cfg_hfp:.5f}")

    # LayerCFG 2-D sweep over s_sem x, s_tex
    best_layer_imgs = None
    best_layer_result = None
    print("\n--- LayerCFG 2-D sweep ---")

    for s_sem in S_SEM_GRID:
        for s_tex in S_TEX_GRID:
            label      = f"LayerCFG sem={s_sem} tex={s_tex}"
            cache_path = f"{RESULTS_DIR}/layer_sem{s_sem}_tex{s_tex}.pt"
            print(f"  {label} …", flush=True)

            if os.path.exists(cache_path):
                layer_imgs = torch.load(cache_path)
                print("    Loaded from disk.")
            else:
                layer_imgs = sample_all_classes(transformer, scheduler, null_class,
                                                SAMPLE_CLASSES, N_PER_CLASS,
                                                method="layer_cfg",
                                                s_sem=s_sem, s_tex=s_tex, f_cut=F_CUT)
                layer_imgs = decode_to_pixels(layer_imgs, vae)
                torch.save(layer_imgs, cache_path)

            logits = logit_extractor.extract(layer_imgs)
            is_mean, is_std = inception_score(logits)
            hfp = compute_hfp(layer_imgs)

            r = {
                "label":   label,
                "method":  "layer_cfg",
                "s_sem":   s_sem,
                "s_tex":   s_tex,
                "hfp":     round(hfp, 5),
                "is_mean": round(is_mean, 3),
                "is_std":  round(is_std, 3),
            }
            results.append(r)
            print(f"    IS={is_mean:.3f}±{is_std:.3f}  HFP={hfp:.5f}")

            # keep the config with the lowest HFP that doesn't tank IS by more than 15%
            is_good_quality = is_mean >= cfg_is_mean * 0.85
            is_better_hfp   = best_layer_result is None or hfp < best_layer_result["hfp"]
            if is_good_quality and is_better_hfp:
                best_layer_result = r
                best_layer_imgs = layer_imgs.clone()

    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n  Saved results.json")

    # print summary table
    print(f"\n{'='*75}")
    print(f"  RESULTS  (ImageNet 256x256, {N_PER_CLASS} imgs/class, {len(SAMPLE_CLASSES)} classes)")
    print(f"{'='*75}")
    print(f"  {'Method':<35}  {'IS':<12}  {'HFP':<10}")
    print(f"  {'-'*65}")
    for r in results:
        print(f"  {r['label']:<35}  {r['is_mean']:.3f}±{r['is_std']:.3f}  {r['hfp']:.5f}")

    best_hfp_result = min((r for r in results if r["method"] == "layer_cfg"), key=lambda r: r["hfp"])
    hfp_reduction = 100 * (cfg_result["hfp"] - best_hfp_result["hfp"]) / cfg_result["hfp"]

    print(f"\n  CFG baseline HFP:         {cfg_result['hfp']:.5f}")
    print(f"  Best LayerCFG HFP:        {best_hfp_result['hfp']:.5f}  ({best_hfp_result['label']})")
    print(f"  HFP reduction:            {hfp_reduction:.1f}%")

    # save all the figures
    if best_layer_imgs is not None:
        save_comparison_grid(
            cfg_imgs, best_layer_imgs,
            SAMPLE_CLASSES,
            label_cfg=f"CFG s={CFG_SCALE} (HFP={cfg_result['hfp']:.4f})",
            label_layer=f"{best_layer_result['label']} (HFP={best_layer_result['hfp']:.4f})",
            n_per_class=N_PER_CLASS,
            save_path=f"{RESULTS_DIR}/figs/comparison_grid.png",
        )

    save_hfp_bar_chart(results, f"{RESULTS_DIR}/figs/hfp_comparison.png")
    save_pareto_plot(results,   f"{RESULTS_DIR}/figs/hfp_vs_is.png")

    elapsed = (time.time() - t0) / 60
    print(f"\n  Total runtime: {elapsed:.1f} min")
    print(f"  All results in {RESULTS_DIR}/")


if __name__ == "__main__":
    main()


Device: cuda
Loading DiT-XL/2-256 …


Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/vae.


  Loaded. null_class=1000, latent_size=32x32
Loading Inception V3 for IS metric…

--- CFG s=4.0 ---
    1/10 classes  ~0.3 min remaining
    2/10 classes  ~0.3 min remaining
    3/10 classes  ~0.2 min remaining
    4/10 classes  ~0.2 min remaining
    5/10 classes  ~0.2 min remaining
    6/10 classes  ~0.1 min remaining
    7/10 classes  ~0.1 min remaining
    8/10 classes  ~0.1 min remaining
    9/10 classes  ~0.0 min remaining
    10/10 classes  ~0.0 min remaining
  IS=2.046±0.076  HFP=0.13947

--- LayerCFG 2-D sweep ---
  LayerCFG sem=1.5 tex=0.1 …
    1/10 classes  ~0.3 min remaining
    2/10 classes  ~0.3 min remaining
    3/10 classes  ~0.2 min remaining
    4/10 classes  ~0.2 min remaining
    5/10 classes  ~0.2 min remaining
    6/10 classes  ~0.1 min remaining
    7/10 classes  ~0.1 min remaining
    8/10 classes  ~0.1 min remaining
    9/10 classes  ~0.0 min remaining
    10/10 classes  ~0.0 min remaining
    IS=3.444±0.616  HFP=0.14071
  LayerCFG sem=1.5 tex=0.25 …
    1/10 

In [9]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Paths
# -----------------------------
EXP3_PATH = f"{RESULTS_DIR}/exp3_comparison.json"
EXP5_PATH = f"{RESULTS_DIR}/exp5_2d_sweep.json"
FIG_DIR   = f"{RESULTS_DIR}/final_report_figs"
os.makedirs(FIG_DIR, exist_ok=True)

# -----------------------------
# Load saved results
# -----------------------------
with open(EXP3_PATH, "r") as f:
    e3 = json.load(f)

with open(EXP5_PATH, "r") as f:
    e5 = json.load(f)

# -----------------------------
# Helper: sort baseline CFG by s
# -----------------------------
def get_cfg_s(label):
    # label format: "CFG s=1.0"
    return float(label.split("s=")[-1])

e3_sorted = sorted(e3, key=lambda r: get_cfg_s(r["label"]))

# -----------------------------
# FIGURE 1:
# Baseline CFG metrics vs guidance scale
# -----------------------------
s_vals   = [get_cfg_s(r["label"]) for r in e3_sorted]
fid_vals = [r["fid"] for r in e3_sorted]
is_vals  = [r["is_mean"] for r in e3_sorted]
ca_vals  = [r["ca"] for r in e3_sorted]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(s_vals, fid_vals, marker='o')
axes[0].set_title("Baseline CFG: FID vs Guidance Scale")
axes[0].set_xlabel("Guidance Scale s")
axes[0].set_ylabel("FID")
axes[0].grid(True, alpha=0.3)

axes[1].plot(s_vals, is_vals, marker='o')
axes[1].set_title("Baseline CFG: IS vs Guidance Scale")
axes[1].set_xlabel("Guidance Scale s")
axes[1].set_ylabel("Inception Score")
axes[1].grid(True, alpha=0.3)

axes[2].plot(s_vals, ca_vals, marker='o')
axes[2].set_title("Baseline CFG: CA vs Guidance Scale")
axes[2].set_xlabel("Guidance Scale s")
axes[2].set_ylabel("Classification Accuracy")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/baseline_cfg_metrics_vs_s.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", f"{FIG_DIR}/baseline_cfg_metrics_vs_s.png")


# -----------------------------
# FIGURE 2:
# LayerCFG FID heatmap over (s_sem, s_tex)
# -----------------------------
s_sem_vals = sorted(list(set(r["s_sem"] for r in e5)))
s_tex_vals = sorted(list(set(r["s_tex"] for r in e5)))

fid_grid = np.zeros((len(s_sem_vals), len(s_tex_vals)))

for i, s_sem in enumerate(s_sem_vals):
    for j, s_tex in enumerate(s_tex_vals):
        match = [r for r in e5 if r["s_sem"] == s_sem and r["s_tex"] == s_tex]
        if len(match) > 0:
            fid_grid[i, j] = match[0]["fid"]
        else:
            fid_grid[i, j] = np.nan

plt.figure(figsize=(7, 5))
im = plt.imshow(fid_grid, cmap="viridis", aspect="auto")
plt.colorbar(im, label="FID")
plt.xticks(ticks=np.arange(len(s_tex_vals)), labels=s_tex_vals)
plt.yticks(ticks=np.arange(len(s_sem_vals)), labels=s_sem_vals)
plt.xlabel("Texture Guidance $s_{tex}$")
plt.ylabel("Semantic Guidance $s_{sem}$")
plt.title("LayerCFG FID Heatmap")

# annotate values
for i in range(len(s_sem_vals)):
    for j in range(len(s_tex_vals)):
        plt.text(j, i, f"{fid_grid[i, j]:.2f}", ha="center", va="center", color="white", fontsize=8)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/layercfg_fid_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", f"{FIG_DIR}/layercfg_fid_heatmap.png")

# -----------------------------
# FIGURE 3:
# Best baseline CFG vs best LayerCFG
# -----------------------------
best_cfg = min(e3, key=lambda r: r["fid"])
best_layer = min(e5, key=lambda r: r["fid"])

metrics = ["fid", "is_mean", "ca", "nhfp"]
titles = ["FID", "Inception Score", "Classification Accuracy", "NHFP"]

cfg_vals = [best_cfg[m] for m in metrics]
layer_vals = [best_layer[m] for m in metrics]

# -----------------------------
# Create 4 side-by-side bar plots
# -----------------------------
fig, axes = plt.subplots(1, 4, figsize=(16,4))

for i, ax in enumerate(axes):

    ax.bar(["CFG", "LayerCFG"],
           [cfg_vals[i], layer_vals[i]],
           color=["steelblue","orange"])

    ax.set_title(titles[i])
    ax.set_ylabel("Value")

    # show numbers on top of bars
    for j, v in enumerate([cfg_vals[i], layer_vals[i]]):
        ax.text(j, v, f"{v:.3f}", ha='center', va='bottom', fontsize=9)

plt.suptitle("Baseline CFG vs LayerCFG", fontsize=14)
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/best_cfg_vs_layercfg_split.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", f"{FIG_DIR}/best_cfg_vs_layercfg_split.png")


# --- Print Summary ---
print("\nBest baseline CFG:")
print(best_cfg)

print("\nBest LayerCFG:")
print(best_layer)

Saved: /content/results/final_report_figs/baseline_cfg_metrics_vs_s.png
Saved: /content/results/final_report_figs/layercfg_fid_heatmap.png
Saved: /content/results/final_report_figs/best_cfg_vs_layercfg_split.png

Best baseline CFG:
{'label': 'CFG s=2.0', 's_sem': 2.0, 's_tex': 2.0, 'fid': 15.49, 'is_mean': 4.846, 'is_std': 0.702, 'ca': 0.9664, 'nhfp': 1.0835, 'precision': 0.762, 'recall': 0.5116}

Best LayerCFG:
{'label': 'Split s_sem=2.0 s_tex=2.0', 's_sem': 2.0, 's_tex': 2.0, 'fid': 15.59, 'is_mean': 4.843, 'is_std': 0.675, 'ca': 0.9772, 'nhfp': 1.0967, 'precision': 0.7468, 'recall': 0.5264}
